# 06 - Business Analytics (Task E04)

**CommerceFlow 360** · Người phụ trách: **Quỳnh Kha** · Mục báo cáo: 4.2, 4.3, 4.4

Chạy 10 truy vấn KPI trong `analytics/sql/analysis_<mart>_<nội dung>.sql` trên 3 Data Mart
(`mart_sales`, `mart_delivery`, `mart_customer`), xuất bảng vào `reports/tables/Bang_4.x_*.csv`
và biểu đồ vào `reports/figures/Hinh_4.x_*.png` dùng cho Chương 4 của báo cáo.

## 1. Chuẩn bị môi trường

In [1]:
# Trên Colab: clone repository (bỏ qua nếu đã đứng trong repository)
import sys, os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not Path('/content/CommerceFlow360').exists():
    !git clone -q https://github.com/huyenpiano1711/CommerceFlow360.git /content/CommerceFlow360
if IN_COLAB:
    %cd /content/CommerceFlow360
print('Google Colab:', IN_COLAB)

Google Colab: False


In [2]:
def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'dbt_project' / 'dbt_project.yml').exists() and (candidate / 'data' / 'raw').is_dir():
            return candidate
    raise FileNotFoundError('Không tìm thấy thư mục gốc CommerceFlow360 (cần có dbt_project/dbt_project.yml).')

PROJECT_ROOT = find_project_root()
DBT_DIR = PROJECT_ROOT / 'dbt_project'
SQL_DIR = PROJECT_ROOT / 'analytics' / 'sql'
TABLES_DIR = PROJECT_ROOT / 'reports' / 'tables'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
DUCKDB_PATH = Path(os.environ.get('DUCKDB_PATH', PROJECT_ROOT / 'commerceflow.duckdb'))
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DUCKDB_PATH:', DUCKDB_PATH)

PROJECT_ROOT: /home/claude/work/CommerceFlow360
DUCKDB_PATH: /home/claude/work/CommerceFlow360/commerceflow.duckdb


In [3]:
%pip install -q "dbt-core>=1.10,<2.0" "dbt-duckdb>=1.10,<2.0" "duckdb>=1.1,<2.0" pandas matplotlib pyarrow

Note: you may need to restart the kernel to use updated packages.


## 2. Build 3 Data Mart (nếu chưa có)

Giả định Bronze/Silver/Gold đã được nạp (xem `notebooks/03_Silver_Transformation.ipynb`,
`notebooks/05_Data_Quality.ipynb`). Cell dưới chỉ build lại `models/gold/marts` để đảm bảo
dữ liệu KPI dùng dưới đây là mới nhất.

In [4]:
import subprocess

def run_dbt(*args, check=True):
    cmd = ['dbt', *args, '--no-use-colors']
    print('$', ' '.join(cmd))
    env = os.environ.copy()
    env['DUCKDB_PATH'] = str(DUCKDB_PATH)
    proc = subprocess.run(cmd, cwd=DBT_DIR, capture_output=True, text=True, env=env)
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print(proc.stderr[-4000:])
        if check:
            raise RuntimeError(f'dbt {args[0]} that bai (exit {proc.returncode})')
    return proc

_ = run_dbt('build', '--select', 'gold.marts')

$ dbt build --select gold.marts --no-use-colors


ART sql table model main.mart_sales .................................. [RUN]
15:35:42  1 of 19 START sql table model main.mart_customer ............................... [RUN]
15:35:43  1 of 19 OK created sql table model main.mart_customer .......................... [OK in 0.72s]
15:35:43  2 of 19 OK created sql table model main.mart_delivery .......................... [OK in 0.74s]
15:35:43  4 of 19 START test not_null_mart_customer_customer_key ......................... [RUN]
15:35:43  5 of 19 START test not_null_mart_customer_customer_unique_id ................... [RUN]
15:35:43  6 of 19 START test not_null_mart_customer_has_valid_order ...................... [RUN]
15:35:43  6 of 19 PASS not_null_mart_customer_has_valid_order ............................ [PASS in 0.17s]
15:35:43  4 of 19 PASS not_null_mart_customer_customer_key ............................... [PASS in 0.19s]
15:35:43  5 of 19 PASS not_null_mart_customer_customer_unique_id ......................... [PASS in 0.19s]
15:3

## 3. E04 - Truy vấn KPI, bảng và biểu đồ

Mỗi mục dưới đây đọc 1 file `.sql` trong `analytics/sql/`, lưu kết quả vào `reports/tables/`
và (nếu phù hợp) vẽ biểu đồ vào `reports/figures/`.

In [5]:
import duckdb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

con = duckdb.connect(str(DUCKDB_PATH), read_only=True)
money_fmt = mticker.FuncFormatter(lambda x, _: f'{x / 1e6:.1f}M')

def run_sql(sql_file):
    sql = (SQL_DIR / sql_file).read_text(encoding='utf-8')
    return con.execute(sql).df()

def save_table(df, name):
    path = TABLES_DIR / name
    df.to_csv(path, index=False)
    print('Bang:', path)
    return df

def save_fig(fig, name):
    path = FIGURES_DIR / name
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()
    plt.close(fig)
    print('Hinh:', path)

### 3.1 Phân tích doanh thu (mart_sales) - mục 4.2

In [6]:
df_sales_month = save_table(run_sql('analysis_sales_monthly_revenue.sql'), 'Bang_4.2.1_DoanhThuTheoThang.csv')
fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(df_sales_month))
ax.plot(x, df_sales_month['revenue'], marker='o', color='#2563eb')
ax.set_xticks(list(x))
ax.set_xticklabels([f"{int(r.year)}-{int(r.month):02d}" for r in df_sales_month.itertuples()], rotation=60, fontsize=7)
ax.yaxis.set_major_formatter(money_fmt)
ax.set_title('Doanh thu theo tháng (mart_sales, đơn hợp lệ)')
ax.set_ylabel('Doanh thu (R$)')
save_fig(fig, 'Hinh_4.2.1_DoanhThuTheoThang.png')
df_sales_month

Bang: /home/claude/work/CommerceFlow360/reports/tables/Bang_4.2.1_DoanhThuTheoThang.csv


Hinh: /home/claude/work/CommerceFlow360/reports/figures/Hinh_4.2.1_DoanhThuTheoThang.png


,year,month,month_name,revenue,order_count,avg_order_value
0,2016,9,September,207.86,2,103.93
1,2016,10,October,44507.30,290,153.47
2,2016,12,December,10.90,1,10.90
3,2017,1,January,120098.27,787,152.60
4,2017,2,February,244959.35,1718,142.58
5,2017,3,March,368341.32,2617,140.75
6,2017,4,April,353842.98,2377,148.86
7,2017,5,May,503159.19,3640,138.23
8,2017,6,June,429916.61,3205,134.14
9,2017,7,July,492287.30,3946,124.76


In [7]:
df_sales_cat = save_table(run_sql('analysis_sales_category_revenue.sql'), 'Bang_4.2.2_Top10DanhMuc.csv')
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(df_sales_cat['product_category'][::-1], df_sales_cat['revenue'][::-1], color='#2563eb')
ax.xaxis.set_major_formatter(money_fmt)
ax.set_title('Top 10 danh mục sản phẩm theo doanh thu')
ax.set_xlabel('Doanh thu (R$)')
save_fig(fig, 'Hinh_4.2.2_Top10DanhMuc.png')
df_sales_cat

Bang: /home/claude/work/CommerceFlow360/reports/tables/Bang_4.2.2_Top10DanhMuc.csv
Hinh: /home/claude/work/CommerceFlow360/reports/figures/Hinh_4.2.2_Top10DanhMuc.png


,product_category,revenue,order_count
0,health_beauty,1255695.13,8800
1,watches_gifts,1198185.21,5604
2,bed_bath_table,1035964.06,9399
3,sports_leisure,979740.92,7673
4,computers_accessories,904322.02,6654
5,furniture_decor,727465.05,6425
6,housewares,626825.80,5847
7,cool_stuff,620770.49,3616
8,auto,586585.73,3872
9,garden_tools,481009.94,3505


In [8]:
df_sales_state = save_table(run_sql('analysis_sales_state_revenue.sql'), 'Bang_4.2.3_Top10BangKhachHang.csv')
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(df_sales_state['customer_state'], df_sales_state['revenue'], color='#2563eb')
ax.yaxis.set_major_formatter(money_fmt)
ax.set_title('Top 10 bang khách hàng theo doanh thu')
ax.set_ylabel('Doanh thu (R$)')
save_fig(fig, 'Hinh_4.2.3_Top10BangKhachHang.png')
df_sales_state

Bang: /home/claude/work/CommerceFlow360/reports/tables/Bang_4.2.3_Top10BangKhachHang.csv


Hinh: /home/claude/work/CommerceFlow360/reports/figures/Hinh_4.2.3_Top10BangKhachHang.png


,customer_state,revenue,order_count,avg_order_value
0,SP,5163867.22,41125,125.57
1,RJ,1811623.42,12697,142.68
2,MG,1573508.20,11496,136.87
3,RS,742559.78,5415,137.13
4,PR,676883.06,4982,135.87
5,SC,518578.28,3599,144.09
6,BA,507108.83,3344,151.65
7,DF,300886.45,2120,141.93
8,GO,287870.46,1998,144.08
9,ES,273532.13,2018,135.55


In [9]:
df_sales_holiday = save_table(run_sql('analysis_sales_holiday_aov.sql'), 'Bang_4.2.4_AOV_NgayLe_vs_NgayThuong.csv')
df_sales_holiday

Bang: /home/claude/work/CommerceFlow360/reports/tables/Bang_4.2.4_AOV_NgayLe_vs_NgayThuong.csv


,is_holiday,revenue,order_count,avg_order_value
0,True,199085.23,1508,132.02
1,False,13295315.51,96691,137.50


### 3.2 Phân tích vận hành giao hàng (mart_delivery) - mục 4.3

In [10]:
df_late_state = save_table(run_sql('analysis_delivery_late_rate_by_state.sql'), 'Bang_4.3.1_TyLeGiaoTreTheoBang.csv')
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(df_late_state['customer_state'], df_late_state['late_rate'] * 100, color='#dc2626')
ax.set_title('Tỷ lệ giao trễ theo bang khách hàng (top 10 đơn nhiều nhất)')
ax.set_ylabel('Tỷ lệ giao trễ (%)')
save_fig(fig, 'Hinh_4.3.1_TyLeGiaoTreTheoBang.png')
df_late_state

Bang: /home/claude/work/CommerceFlow360/reports/tables/Bang_4.3.1_TyLeGiaoTreTheoBang.csv


Hinh: /home/claude/work/CommerceFlow360/reports/figures/Hinh_4.3.1_TyLeGiaoTreTheoBang.png


,customer_state,order_count,late_rate,avg_days_late
0,SP,40501,0.0449,0.37
1,RJ,12350,0.1211,1.64
2,MG,11354,0.0457,0.38
3,RS,5345,0.0608,0.62
4,PR,4923,0.0404,0.34
5,SC,3546,0.0821,0.68
6,BA,3256,0.1216,1.46
7,DF,2080,0.0567,0.42
8,ES,1995,0.1073,1.21
9,GO,1957,0.0654,0.74


In [11]:
df_late_seller = save_table(run_sql('analysis_delivery_late_rate_by_seller.sql'), 'Bang_4.3.2_TopSellerGiaoTre.csv')
df_late_seller

Bang: /home/claude/work/CommerceFlow360/reports/tables/Bang_4.3.2_TopSellerGiaoTre.csv


,seller_id,order_count,late_rate,avg_review_score
0,ede0c03645598cdfc63ca8237acbe73d,43,0.3256,3.57
1,ad781527c93d00d89a11eecd9dcad7c1,38,0.3158,3.34
2,54965bbe3e4f07ae045b90b0b8541f52,73,0.3014,3.14
3,2a1348e9addc1af5aaa619b1a3679d6b,48,0.2708,3.17
4,835f0f7810c76831d6c7d24c7a646d4d,42,0.2619,3.36
5,beadbee30901a7f61d031b6b686095ad,64,0.2344,3.91
6,a49928bcdf77c55c6d6e05e09a9b4ca5,96,0.2188,3.03
7,ef990a83bbea832f36ebe81376335aa8,43,0.2093,3.63
8,712e6ed8aa4aa1fa65dab41fed5737e4,77,0.2078,3.39
9,3fd1e727ba94cfe122d165e176ce7967,35,0.2000,3.94


In [12]:
df_review_ontime = save_table(run_sql('analysis_delivery_review_by_ontime.sql'), 'Bang_4.3.3_DiemDanhGia_DungHan_vs_Tre.csv')
fig, ax = plt.subplots(figsize=(5, 4.5))
ax.bar(df_review_ontime['nhom_giao_hang'], df_review_ontime['avg_review_score'], color=['#16a34a', '#dc2626'])
ax.set_ylim(0, 5)
ax.set_title('Điểm đánh giá trung bình: đúng hạn vs trễ')
ax.set_ylabel('Điểm đánh giá (1-5)')
save_fig(fig, 'Hinh_4.3.2_DiemDanhGia_DungHan_vs_Tre.png')
df_review_ontime

Bang: /home/claude/work/CommerceFlow360/reports/tables/Bang_4.3.3_DiemDanhGia_DungHan_vs_Tre.csv


Hinh: /home/claude/work/CommerceFlow360/reports/figures/Hinh_4.3.2_DiemDanhGia_DungHan_vs_Tre.png


,nhom_giao_hang,order_count,avg_review_score
0,đúng hạn,89443,4.29
1,giao trễ,6381,2.27


### 3.3 Phân tích khách hàng (mart_customer) - mục 4.4

In [13]:
df_repeat = save_table(run_sql('analysis_customer_repeat_rate.sql'), 'Bang_4.4.1_TyLeMuaLai.csv')
fig, ax = plt.subplots(figsize=(5, 4.5))
ax.bar(df_repeat['nhom_khach_hang'], df_repeat['pct_customers'], color=['#2563eb', '#f59e0b'])
ax.set_title('Tỷ lệ khách mua 1 lần vs mua lại')
ax.set_ylabel('% khách hàng')
save_fig(fig, 'Hinh_4.4.1_TyLeMuaLai.png')
df_repeat

Bang: /home/claude/work/CommerceFlow360/reports/tables/Bang_4.4.1_TyLeMuaLai.csv


Hinh: /home/claude/work/CommerceFlow360/reports/figures/Hinh_4.4.1_TyLeMuaLai.png


,nhom_khach_hang,customer_count,pct_customers,avg_total_spend
0,mua lại (>=2 đơn),2888,3.04,308.36
1,mua 1 lần,92102,96.96,161.22


In [14]:
df_segment = save_table(run_sql('analysis_customer_spend_segments.sql'), 'Bang_4.4.2_PhanKhucChiTieu.csv')
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.bar(df_segment['spend_quartile'].astype(str), df_segment['pct_revenue'], color='#2563eb')
ax.set_title('% tổng doanh thu theo phân khúc chi tiêu (quartile)')
ax.set_xlabel('Quartile (1 = chi tiêu thấp nhất)')
ax.set_ylabel('% tổng chi tiêu')
save_fig(fig, 'Hinh_4.4.2_PhanKhucChiTieu.png')
df_segment

Bang: /home/claude/work/CommerceFlow360/reports/tables/Bang_4.4.2_PhanKhucChiTieu.csv


Hinh: /home/claude/work/CommerceFlow360/reports/figures/Hinh_4.4.2_PhanKhucChiTieu.png


,spend_quartile,customer_count,min_spend,max_spend,total_spend_segment,pct_revenue
0,1,23748,9.59,63.10,1036556.95,6.59
1,2,23747,63.10,107.90,1987313.44,12.63
2,3,23747,107.92,182.94,3354705.92,21.31
3,4,23747,182.95,13664.08,9360560.70,59.47


In [15]:
df_recency = save_table(run_sql('analysis_customer_recency.sql'), 'Bang_4.4.3_PhanBoRecency.csv')
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.bar(df_recency['nhom_recency'], df_recency['pct_customers'], color='#2563eb')
ax.set_title('Phân bố thời gian từ lần mua gần nhất (recency)')
ax.set_ylabel('% khách hàng')
save_fig(fig, 'Hinh_4.4.3_PhanBoRecency.png')
df_recency

Bang: /home/claude/work/CommerceFlow360/reports/tables/Bang_4.4.3_PhanBoRecency.csv


Hinh: /home/claude/work/CommerceFlow360/reports/figures/Hinh_4.4.3_PhanBoRecency.png


,nhom_recency,customer_count,pct_customers
0,0-90 ngay,9463,9.96
1,91-180 ngay,18234,19.20
2,181-365 ngay,39348,41.42
3,> 365 ngay,27945,29.42


## 4. Kiểm tra đầu ra

Xác nhận đủ 9 bảng + 7 hình đã xuất ra `reports/` trước khi đưa vào báo cáo Chương 4.

In [16]:
expected_tables = [
    'Bang_4.2.1_DoanhThuTheoThang.csv', 'Bang_4.2.2_Top10DanhMuc.csv',
    'Bang_4.2.3_Top10BangKhachHang.csv', 'Bang_4.2.4_AOV_NgayLe_vs_NgayThuong.csv',
    'Bang_4.3.1_TyLeGiaoTreTheoBang.csv', 'Bang_4.3.2_TopSellerGiaoTre.csv',
    'Bang_4.3.3_DiemDanhGia_DungHan_vs_Tre.csv',
    'Bang_4.4.1_TyLeMuaLai.csv', 'Bang_4.4.2_PhanKhucChiTieu.csv', 'Bang_4.4.3_PhanBoRecency.csv',
]
expected_figures = [
    'Hinh_4.2.1_DoanhThuTheoThang.png', 'Hinh_4.2.2_Top10DanhMuc.png', 'Hinh_4.2.3_Top10BangKhachHang.png',
    'Hinh_4.3.1_TyLeGiaoTreTheoBang.png', 'Hinh_4.3.2_DiemDanhGia_DungHan_vs_Tre.png',
    'Hinh_4.4.1_TyLeMuaLai.png', 'Hinh_4.4.2_PhanKhucChiTieu.png', 'Hinh_4.4.3_PhanBoRecency.png',
]
checks = {
    'du 10 bang Chuong 4': all((TABLES_DIR / f).is_file() for f in expected_tables),
    'du 8 hinh Chuong 4': all((FIGURES_DIR / f).is_file() for f in expected_figures),
}
import pandas as pd
display(pd.Series(checks).rename('PASS'))
assert all(checks.values()), 'Thieu bang/hinh - xem lai cac cell phia tren.'

du 10 bang Chuong 4    True
du 8 hinh Chuong 4     True
Name: PASS, dtype: bool